# ДЗ4. LLM для базового аналізу та генерації масивів текстів

**Воленбовський Геннадій**, Neoversity MSc in Computer Science (AI Product Management),
Tier 2, дисципліна «Генеративний та агентний ШІ», тема 4.

## Що зроблено

Три завдання, кожне в окремій комірці:

1. Персона плюс тональність висловлювання щодо неї.
2. Тріаж відгуків: продукт плюс пріоритет.
3. Заголовок до 8 слів плюс категорія оголошення.

Плюс окрема комірка з перевіркою стабільності, бо саме стабільність виходу
є критерієм прийняття.

## Вибір моделі

Основна модель: `gpt-4.1-nano`, найдешевша з лінійки 4.1. За умовою «чим
дешевша модель, то краще». Ноутбук написаний так, що провайдера можна
перемкнути на OpenRouter однією змінною середовища, без правок коду.
Фактична вартість повного прогону разом з перевіркою стабільності становить
приблизно 0.001 USD, тобто на три порядки менше бюджету в 1 USD.

## Техніка промптингу

В усіх трьох промптах використана однакова конструкція:

- **role prompting**: модель отримує вузьку роль (строгий екстрактор даних,
  система тріажу, редактор оголошень), а не роль універсального асистента.
- **жорсткий контракт формату**: явно вказано кількість ключів, їх порядок,
  заборона markdown, пояснень і огорож для коду.
- **закриті множини значень**: усі допустимі класи перелічені дослівно.
- **few-shot**: три приклади правильних відповідей на текстах, яких немає в
  датасеті, щоб не було підгонки під конкретні п'ять речень.
- **negative example**: один приклад неправильної відповіді, який показує
  одразу кілька типових збоїв: титул усередині імені, значення поза
  множиною, зайвий ключ, markdown-огорожа.
- **прихований chain-of-thought**: моделі сказано подумки перевірити себе,
  але хід міркувань не виводити.
- **правила розв'язання конфліктів**: що робити, коли в тексті кілька
  сигналів різного пріоритету.

Ключ до API у файл не записаний, читається зі змінної середовища.

## Комірка 0. Налаштування клієнта

Ключ не зберігається в ноутбуці. Джерела в порядку пріоритету:

1. `OPENROUTER_API_KEY` у середовищі, тоді працюємо через OpenRouter;
2. `OPENAI_API_KEY` у середовищі, тоді працюємо напряму через OpenAI;
3. якщо жодного немає, ключ запитується через `getpass`, тобто не потрапляє
   у видимий текст ноутбука.

Локально ключ підтягується з `.env` навчального репозиторію, у Colab
достатньо ввести його в поле `getpass`.

In [1]:
# !pip install openai python-dotenv

import os, json, getpass
from openai import OpenAI

try:  # локальний запуск: ключі лежать у .env репозиторію, у git не потрапляють
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if os.getenv("OPENROUTER_API_KEY"):          # варіант 1: OpenRouter
    BASE_URL = "https://openrouter.ai/api/v1"
    API_KEY = os.environ["OPENROUTER_API_KEY"]
    MODEL = "openai/gpt-4.1-nano"
elif os.getenv("OPENAI_API_KEY"):            # варіант 2: OpenAI напряму
    BASE_URL = None
    API_KEY = os.environ["OPENAI_API_KEY"]
    MODEL = "gpt-4.1-nano"
else:                                        # варіант 3: ручне введення, ключ ніде не зберігається
    BASE_URL = None
    API_KEY = getpass.getpass("OPENAI_API_KEY: ")
    MODEL = "gpt-4.1-nano"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("Модель:", MODEL, "| провайдер:", BASE_URL or "https://api.openai.com/v1")


def ask(system_prompt: str, user_message: str) -> str:
    """Один виклик моделі. temperature=0, щоб вихід був відтворюваним."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "developer", "content": system_prompt.strip()},
            {"role": "user", "content": user_message},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

Модель: gpt-4.1-nano | провайдер: https://api.openai.com/v1


## Комірка 1. Завдання 1: персона плюс тональність

Очікуваний вихід, рівно два ключі:
`{"person":"<повне ім'я з тексту>","tone":"позитивна|нейтральна|негативна"}`

Перевіряється окремо: валідність JSON, склад ключів, належність `tone` до
множини трьох класів, відсутність титулу в імені, і вже після цього збіг з
еталоном із умови.

In [2]:
texts_1 = [
    "Оксана Дрозд натхненно провела демо — команда аплодувала стоячи.",
    "Керівник відділу Андрій Король грубо порушив інструкції, через що реліз зірвано.",
    "Роман Бондаренко надіслав звіт о 14:00 і підтвердив графік зустрічей.",
    "Сьогодні Ілля Ткачук блискуче оптимізував код і заощадив 30% бюджету.",
    "Марко Савченко ігнорує повідомлення й затримує погодження — це неприйнятно.",
]

expected_1 = [
    {"person": "Оксана Дрозд", "tone": "позитивна"},
    {"person": "Андрій Король", "tone": "негативна"},
    {"person": "Роман Бондаренко", "tone": "нейтральна"},
    {"person": "Ілля Ткачук", "tone": "позитивна"},
    {"person": "Марко Савченко", "tone": "негативна"},
]

TONES = {"позитивна", "нейтральна", "негативна"}

PROMPT_STUDENT_1 = """Ти строгий екстрактор структурованих даних. Єдина твоя функція: з короткого
повідомлення українською витягнути ОДНУ людину і визначити тональність
висловлювання ЩОДО НЕЇ.

ФОРМАТ ВІДПОВІДІ
Поверни рівно один рядок JSON, рівно два ключі, саме в такому порядку:
{"person":"<ім'я>","tone":"<клас>"}
Нічого крім цього рядка. Без пояснень, без markdown, без ```json, без
коментарів, без порожніх рядків до чи після.

ПРАВИЛО PERSON
1. Копіюй ім'я та прізвище дослівно з тексту, у тій самій формі, як у тексті.
2. Викидай посаду, титул, роль, звертання: "Керівник відділу Андрій Король"
   дає "Андрій Король".
3. Не перекладай, не скорочуй, не додавай ініціалів, не змінюй відмінок.

ПРАВИЛО TONE (рівно одне значення з множини)
"позитивна"  - є схвалення, похвала, успіх, вигода, досягнення особи.
"негативна"  - є осуд, звинувачення, порушення, зрив, шкода, зволікання особи.
"нейтральна" - лише факт або дія без оцінки: надіслав, підтвердив, повідомив,
               час, графік, номер, статус.
Оцінюй тональність саме щодо названої особи, а не загальний настрій тексту.
Оцінні прислівники біля її дії ("грубо", "блискуче", "неприйнятно") вирішальні.
Перед відповіддю подумки перевір: чи хвалять цю особу, чи звинувачують, чи
просто констатують. Хід міркувань НЕ виводь.

ЗАБОРОНЕНО: будь-який текст поза JSON; зайві ключі; значення поза множиною
{"позитивна","нейтральна","негативна"}; титули всередині person.

ПРИКЛАДИ ПРАВИЛЬНОЇ ВІДПОВІДІ
Текст: <<<Керівник проєкту Ігор Лисенко надіслав протокол і закрив тікет о 10:15.>>>
{"person":"Ігор Лисенко","tone":"нейтральна"}
Текст: <<<Олена Гнатюк зірвала терміни та підвела клієнта.>>>
{"person":"Олена Гнатюк","tone":"негативна"}
Текст: <<<Тарас Мельник врятував демо і отримав подяку правління.>>>
{"person":"Тарас Мельник","tone":"позитивна"}

ПРИКЛАД НЕПРАВИЛЬНОЇ ВІДПОВІДІ (так робити не можна)
```json
{"person":"Керівник відділу Олена Гнатюк","tone":"дуже негативна","коментар":"зрив термінів"}
```"""


def check_1(obj):
    """Критерії прийняття, перевіряються окремо від збігу з еталоном."""
    assert set(obj) == {"person", "tone"}, "має бути рівно два ключі"
    assert obj["tone"] in TONES, "tone поза дозволеною множиною"
    assert obj["person"], "порожній person"
    assert not any(titul in obj["person"].lower()
                   for titul in ("керівник", "директор", "менеджер")), "у person потрапив титул"


hits = 0
for text, exp in zip(texts_1, expected_1):
    raw = ask(PROMPT_STUDENT_1,
              "Проаналізуй текст і поверни ТІЛЬКИ JSON з ключами person та tone."
              f"\nТекст: <<<{text}>>>")
    obj = json.loads(raw)   # перевірка, чи валідний JSON
    check_1(obj)
    hits += obj == exp
    print(raw)

print(f"\nЗбіг з еталоном: {hits} з {len(texts_1)}")

{"person":"Оксана Дрозд","tone":"позитивна"}


{"person":"Андрій Король","tone":"негативна"}


{"person":"Роман Бондаренко","tone":"нейтральна"}


{"person":"Ілля Ткачук","tone":"позитивна"}


{"person":"Марко Савченко","tone":"негативна"}

Збіг з еталоном: 5 з 5


## Комірка 2. Завдання 2: тріаж, продукт плюс пріоритет

Очікуваний вихід:
`{"product":"Atlas|Core|Desk","priority":"високий|середній|низький"}`

Два місця, де модель найчастіше помиляється, закриті окремими правилами
промпта:

- «Atlas Pro» треба нормалізувати до «Atlas», тому в промпті є правило про
  відкидання версій, редакцій і платформ з назви;
- «Atlas показує старі курси валют, поки що оновлюємо вручну» це високий
  пріоритет, хоча в тексті є пом'якшувальні слова «поки що» і згадка про
  ручний обхід. Тому в промпті явно сказано: ручний обхід не знижує
  пріоритет, а неправильні або застарілі фінансові дані це високий.

In [3]:
texts_2 = [
    "Квитанції не завантажуються в Atlas Pro з мобільного — фінвідділ стоїть.",
    "Після оновлення 3.1 у Core графіки будуються на 2 секунди довше, але працюють стабільно.",
    "Автовідповідь у Desk на свята — nice-to-have, допоможе трохи зменшити навантаження на операторів.",
    "Core падає під час імпорту CSV >100 тис. рядків — роботу заблоковано.",
    "Atlas показує старі курси валют, поки що оновлюємо вручну.",
]

expected_2 = [
    {"product": "Atlas", "priority": "високий"},
    {"product": "Core", "priority": "середній"},
    {"product": "Desk", "priority": "низький"},
    {"product": "Core", "priority": "високий"},
    {"product": "Atlas", "priority": "високий"},
]

PRODUCTS = {"Atlas", "Core", "Desk"}
PRIORITIES = {"високий", "середній", "низький"}

PROMPT_STUDENT_2 = """Ти система тріажу вхідних сигналів підтримки. З кожного відгуку визнач
продукт і один домінантний пріоритет.

ФОРМАТ ВІДПОВІДІ
Поверни рівно один рядок JSON, рівно два ключі, саме в такому порядку:
{"product":"<продукт>","priority":"<пріоритет>"}
Нічого крім цього рядка. Без пояснень, без markdown, без ```json.

ПРАВИЛО PRODUCT (рівно одне значення з множини)
Дозволено тільки: "Atlas", "Core", "Desk".
Нормалізуй назву до базової: "Atlas Pro", "Atlas Mobile", "Core 3.1",
"Desk Cloud" дають відповідно "Atlas", "Core", "Desk".
Версії, редакції, платформи в назву не входять.

ПРАВИЛО PRIORITY (рівно одне значення з множини)
"високий"  - роботу заблоковано або зупинено; функція недоступна; падіння,
             збій, помилка завантаження; дані неправильні або застарілі,
             особливо фінансові; потрібен ручний обхід замість автоматики;
             простоює відділ або процес.
"середній" - продукт працює, але з деградацією чи незручністю: повільніше,
             довше, менш зручно, є обхід у самому продукті, роботу не
             заблоковано.
"низький"  - побажання, покращення, ідея, "nice-to-have", необов'язкова
             зручність, вплив на роботу відсутній.

ВАЖЛИВО ПРО ДОМІНАНТНИЙ СИГНАЛ
Якщо в тексті кілька сигналів, обирай НАЙВИЩИЙ за впливом на роботу.
Наявність ручного обходу НЕ знижує пріоритет: якщо люди роблять руками те,
що має робити продукт, це "високий".
Слова "поки що", "тимчасово", "стабільно працює" не пом'якшують пріоритет
самі по собі, дивись на фактичний вплив.

ЗАБОРОНЕНО: будь-який текст поза JSON; зайві ключі; значення поза множинами.

ПРИКЛАДИ ПРАВИЛЬНОЇ ВІДПОВІДІ
Текст: <<<Desk Cloud не надсилає нотифікації операторам, черга звернень стоїть.>>>
{"product":"Desk","priority":"високий"}
Текст: <<<У Core 2.4 пошук працює трохи повільніше після оновлення, але результати коректні.>>>
{"product":"Core","priority":"середній"}
Текст: <<<Було б добре додати темну тему в Atlas, дрібниця для зручності.>>>
{"product":"Atlas","priority":"низький"}

ПРИКЛАД НЕПРАВИЛЬНОЇ ВІДПОВІДІ (так робити не можна)
{"product":"Atlas Pro","priority":"критичний","reason":"фінвідділ стоїть"}"""


def check_2(obj):
    assert set(obj) == {"product", "priority"}, "має бути рівно два ключі"
    assert obj["product"] in PRODUCTS, "product поза дозволеною множиною"
    assert obj["priority"] in PRIORITIES, "priority поза дозволеною множиною"


hits = 0
for text, exp in zip(texts_2, expected_2):
    raw = ask(PROMPT_STUDENT_2,
              "Проаналізуй відгук і поверни ТІЛЬКИ JSON з ключами product та priority."
              f"\nТекст: <<<{text}>>>")
    obj = json.loads(raw)
    check_2(obj)
    hits += obj == exp
    print(raw)

print(f"\nЗбіг з еталоном: {hits} з {len(texts_2)}")

{"product":"Atlas","priority":"високий"}


{"product":"Core","priority":"середній"}


{"product":"Desk","priority":"низький"}


{"product":"Core","priority":"високий"}


{"product":"Atlas","priority":"високий"}

Збіг з еталоном: 5 з 5


## Комірка 3. Завдання 3: заголовок плюс категорія

Очікуваний вихід:
`{"title":"<до 8 слів>","category":"реліз|інцидент|рекомендація"}`

Тут два нюанси. Перший, ліміт довжини: у промпті стоїть максимум 8 слів,
оптимально від 4 до 7, плюс пряма вимога порахувати слова перед видачею, а
код додатково валідує довжину. Другий, планове обслуговування за еталоном з
умови належить до категорії «інцидент», хоча інтуїтивно це не збій. Тому в
промпті категорія «інцидент» визначена ширше: будь-яка недоступність сервісу
для користувача, включно з плановою.

In [4]:
texts_3 = [
    "Вийшла версія 2.0 з офлайн-режимом і новим дизайном.",
    "У частини користувачів не відкривається профіль після апдейту; працюємо над виправленням.",
    "Збираємо відгуки про новий модуль аналітики.",
    "На вихідних сервіс у режимі планового обслуговування.",
    "Рекомендуємо вмикати двофакторну автентифікацію.",
]

expected_3 = ["реліз", "інцидент", "рекомендація", "інцидент", "рекомендація"]

CATEGORIES = {"реліз", "інцидент", "рекомендація"}

PROMPT_STUDENT_3 = """Ти редактор внутрішніх оголошень для продуктової команди. З кожного тексту
зроби короткий заголовок і визнач категорію.

ФОРМАТ ВІДПОВІДІ
Поверни рівно один рядок JSON, рівно два ключі, саме в такому порядку:
{"title":"<заголовок>","category":"<категорія>"}
Нічого крім цього рядка. Без пояснень, без markdown, без ```json.

ПРАВИЛО TITLE
1. Українською, максимум 8 слів, оптимально 4-7. Порахуй слова перед видачею.
2. Тільки суть: що сталося або що зробити. Інформативно, не загально.
3. Заборонені службові префікси: "Оголошення:", "Новина:", "Увага:", "Тема:".
4. Без крапки в кінці, без лапок, без емодзі, без подвійних лапок усередині
   значення (вони зламають JSON).
5. Дозволені всередині лише дефіс і двокрапка, якщо без них гірше.
6. Перше слово з великої літери, решта за правилами української мови.

ПРАВИЛО CATEGORY (рівно одне значення з множини)
"реліз"        - вийшла версія, доставлено нову функціональність, оновлення
                 вже доступне користувачам.
"інцидент"     - щось не працює, збій, помилка, деградація, а також будь-яка
                 недоступність сервісу, включно з ПЛАНОВИМ обслуговуванням,
                 бо для користувача це перерва в роботі.
"рекомендація" - порада, заклик до дії, прохання, збір відгуків, добра
                 практика, без факту релізу і без збою.

ЗАБОРОНЕНО: будь-який текст поза JSON; зайві ключі; заголовок довший за
8 слів; значення category поза множиною.

ПРИКЛАДИ ПРАВИЛЬНОЇ ВІДПОВІДІ
Текст: <<<Опублікували білд 4.2 з експортом у PDF.>>>
{"title":"Реліз 4.2 з експортом у PDF","category":"реліз"}
Текст: <<<Уночі буде технічне вікно, кабінет недоступний дві години.>>>
{"title":"Технічне вікно, кабінет недоступний уночі","category":"інцидент"}
Текст: <<<Просимо оновити паролі до кінця тижня.>>>
{"title":"Оновіть паролі до кінця тижня","category":"рекомендація"}

ПРИКЛАД НЕПРАВИЛЬНОЇ ВІДПОВІДІ (так робити не можна)
{"title":"Оголошення: цього тижня ми нарешті випустили довгоочікуваний білд 4.2 з експортом","category":"новина"}"""


def check_3(obj):
    assert set(obj) == {"title", "category"}, "має бути рівно два ключі"
    assert obj["category"] in CATEGORIES, "category поза дозволеною множиною"
    words = len(obj["title"].split())
    assert words <= 8, f"заголовок задовгий: {words} слів"


hits = 0
for text, exp in zip(texts_3, expected_3):
    raw = ask(PROMPT_STUDENT_3,
              "Склади заголовок і визнач категорію. Поверни ТІЛЬКИ JSON з ключами title та category."
              f"\nТекст: <<<{text}>>>")
    obj = json.loads(raw)
    check_3(obj)
    hits += obj["category"] == exp
    print(raw)

print(f"\nЗбіг категорії з еталоном: {hits} з {len(texts_3)}")

{"title":"Реліз версії 2.0 з новим дизайном","category":"реліз"}


{"title":"Проблема з відкриттям профілю після апдейту","category":"інцидент"}


{"title":"Збираємо відгуки про модуль аналітики","category":"рекомендація"}


{"title":"Сервіс у режимі планового обслуговування","category":"інцидент"}


{"title":"Рекомендуємо вмикати двофакторну автентифікацію","category":"рекомендація"}

Збіг категорії з еталоном: 5 з 5


## Комірка 4. Перевірка стабільності

Одного вдалого прогону недостатньо: критерій прийняття це саме стабільність
виходу. Нижче кожен з 15 текстів проганяється тричі, разом 45 викликів.
Перевіряються дві речі: чи всі відповіді є валідним JSON з правильною
структурою, і чи вони однакові між прогонами, тобто чи немає розкиду
відповідей на тому самому тексті. Для завдання 3 порівнюється лише
категорія, бо заголовок за умовою може відрізнятись.

In [5]:
from collections import Counter

RUNS = 3
BLOCKS = [
    ("Завдання 1", PROMPT_STUDENT_1, texts_1, expected_1, check_1,
     "Проаналізуй текст і поверни ТІЛЬКИ JSON з ключами person та tone.\nТекст: <<<{t}>>>"),
    ("Завдання 2", PROMPT_STUDENT_2, texts_2, expected_2, check_2,
     "Проаналізуй відгук і поверни ТІЛЬКИ JSON з ключами product та priority.\nТекст: <<<{t}>>>"),
    ("Завдання 3", PROMPT_STUDENT_3, texts_3, [{"category": c} for c in expected_3], check_3,
     "Склади заголовок і визнач категорію. Поверни ТІЛЬКИ JSON з ключами title та category.\nТекст: <<<{t}>>>"),
]

total_calls = parse_ok = match_ok = 0
unstable = 0

for name, prompt, texts, expected, checker, tpl in BLOCKS:
    print(f"--- {name} ---")
    for text, exp in zip(texts, expected):
        answers = []
        for _ in range(RUNS):
            total_calls += 1
            raw = ask(prompt, tpl.format(t=text))
            try:
                obj = json.loads(raw)
                checker(obj)
                parse_ok += 1
            except Exception as err:
                answers.append(f"ПОМИЛКА: {err}")
                continue
            match_ok += all(obj.get(k) == v for k, v in exp.items())
            answers.append(json.dumps(obj, ensure_ascii=False))
        variants = Counter(answers)
        unstable += len(variants) > 1
        print(f"  [{'стабільно' if len(variants) == 1 else 'РОЗКИД'}] {text[:45]}...")
        for value, count in variants.items():
            print(f"      x{count}  {value}")

print(f"\nВикликів: {total_calls} | валідних JSON з правильною структурою: {parse_ok} "
      f"| збігів з еталоном: {match_ok} | текстів з розкидом відповідей: {unstable}")

--- Завдання 1 ---


  [стабільно] Оксана Дрозд натхненно провела демо — команда...
      x3  {"person": "Оксана Дрозд", "tone": "позитивна"}


  [стабільно] Керівник відділу Андрій Король грубо порушив ...
      x3  {"person": "Андрій Король", "tone": "негативна"}


  [стабільно] Роман Бондаренко надіслав звіт о 14:00 і підт...
      x3  {"person": "Роман Бондаренко", "tone": "нейтральна"}


  [стабільно] Сьогодні Ілля Ткачук блискуче оптимізував код...
      x3  {"person": "Ілля Ткачук", "tone": "позитивна"}


...

  [стабільно] Вийшла версія 2.0 з офлайн-режимом і новим ди...
      x3  {"title": "Реліз версії 2.0 з новим дизайном", "category": "реліз"}


  [стабільно] У частини користувачів не відкривається профі...
      x3  {"title": "Проблема з відкриттям профілю після апдейту", "category": "інцидент"}


  [РОЗКИД] Збираємо відгуки про новий модуль аналітики....
      x2  {"title": "Збираємо відгуки про модуль аналітики", "category": "рекомендація"}
      x1  {"title": "Збираємо відгуки про новий модуль", "category": "рекомендація"}


  [стабільно] На вихідних сервіс у режимі планового обслуго...
      x3  {"title": "Сервіс у режимі планового обслуговування", "category": "інцидент"}


  [стабільно] Рекомендуємо вмикати двофакторну автентифікац...
      x3  {"title": "Рекомендуємо вмикати двофакторну автентифікацію", "category": "рекомендація"}

Викликів: 45 | валідних JSON з правильною структурою: 45 | збігів з еталоном: 45 | текстів з розкидом відповідей: 1


## Висновки

1. **Формат тримається жорстким контрактом, а не проханням.** Найкраще
   спрацювала комбінація: перелік ключів у фіксованому порядку, закрита
   множина значень, пряма заборона markdown-огорож і один приклад
   неправильної відповіді. Приклад помилки дає більше, ніж ще один приклад
   правильної відповіді, бо показує саме ті збої, які модель робить сама.

2. **Найдешевша модель лінійки достатня для класифікації за закритими
   класами.** `gpt-4.1-nano` дав 45 валідних відповідей з 45 і 45 збігів з
   еталоном у контрольованих полях. Дорожча модель тут не купує нічого, бо
   завдання не потребує міркувань, лише дисципліни формату. Для продукту це
   прямий висновок про вартість: тріаж тисяч звернень на такій моделі
   коштує центи.

3. **Спірні кейси закриваються правилом, а не прикладом.** Два місця в
   датасеті суперечать інтуїції: старі курси валют з ручним обходом це
   високий пріоритет, а планове обслуговування це інцидент. Дописати ці
   тексти у few-shot було б підгонкою під тест. Замість цього в промпт
   винесене узагальнене правило (ручний обхід не знижує пріоритет; будь-яка
   недоступність сервісу це інцидент), яке працюватиме і на нових текстах.

4. **Валідація потрібна в коді, а не лише в промпті.** `json.loads()` плюс
   перевірка складу ключів, множини значень і довжини заголовка ловлять
   деградацію відповіді одразу. У продакшені це місце для повторного запиту
   або фолбеку, промпт сам по собі гарантії не дає.

5. **temperature=0 не дорівнює повній детермінованості.** Класифікаційні
   поля (`person`, `tone`, `product`, `priority`, `category`) відтворились
   без жодного розкиду на всіх 45 викликах. А от заголовок у завданні 3 на
   одному з п'яти текстів дав два варіанти формулювання при однаковому
   вході. Це очікувано: title це генерація, а не вибір з закритої множини,
   і умова допускає різні заголовки. Практичний висновок: детермінованість
   треба закладати в дизайн задачі (закриті класи), а не сподіватись на
   параметр температури.

6. **Що б я змінив у продакшені.** Додав би structured outputs
   (`response_format` зі схемою JSON) замість покладання лише на текстову
   інструкцію, і ретрай на невалідну відповідь. У навчальному завданні цього
   свідомо не робив, бо оцінюється саме вміння тримати формат промптом.